In [1]:
!nvidia-smi
!pip install ultralytics

Sun Sep 13 10:41:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import cv2
import random

# A function to search for dataset path
def find_dataset_root():
    input_root = '/kaggle/input/'
    detected_path = None
    for root, dirs, files in os.walk(input_root):
        if 'train' in dirs and 'val' in dirs:
            detected_path = root
            break
    return detected_path

REAL_BASE_PATH = find_dataset_root()
print(f"Dataset Path: {REAL_BASE_PATH}")

Dataset Path: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data


In [3]:
import yaml

if REAL_BASE_PATH:
    data_config = {
        'path': REAL_BASE_PATH,
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images' if os.path.exists(os.path.join(REAL_BASE_PATH, 'test')) else 'val/images',
        'names': {0: 'Smoke', 1: 'Fire'} # Here a dictionary to avoid the old error by me
    }

    with open('/kaggle/working/data.yaml', 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)
    print("data.yaml has been created successfully, with the true classes names") 
    # The main error was in data card(from kaggle source)
    print(data_config['names'])   # To show real 'names' value
    print("0:smoke, 1:Fire")

data.yaml has been created successfully, with the true classes names
{0: 'Smoke', 1: 'Fire'}
0:smoke, 1:Fire


In [4]:
from ultralytics import YOLO

model = YOLO("/kaggle/input/datasets/issahasan43/yolo26m-best-pt-epoch64-final/best.pt")

test_metrics = model.val(
    data="/kaggle/working/data.yaml",
    split="test",
    imgsz=640,
    batch=32,
    cache=True,
    device=[-1, -1]  
)

print("\n" + "="*40 + "\n GLOBAL METRICS \n" + "="*40)
print("mAP50-95:", test_metrics.box.map)          # mAP at IoU 0.50:0.95
print("mAP50:", test_metrics.box.map50)          # mAP at IoU 0.50
print("mAP75:", test_metrics.box.map75)          # mAP at IoU 0.75
print("Mean precision:", test_metrics.box.mp)
print("Mean recall:", test_metrics.box.mr)
print("Fitness:", test_metrics.box.fitness())    # Weighted score for model selection

print("\n" + "="*40 + "\n PER-CLASS METRICS \n" + "="*40)
print("Class indices evaluated:", test_metrics.box.ap_class_index)
print("Per-class mAP50-95:", test_metrics.box.maps)

print("\n" + "="*40 + "\nTIMING \n" + "="*40)

# Per-stage timing breakdown in milliseconds per image
print("Timing breakdown (ms/image):", test_metrics.speed)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Searching for 2 idle GPUs with free memory >= 20.0% and free utilization >= 0.0%...
Selected idle CUDA devices [1, 0]
Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:1 (Tesla T4, 14912MiB)
                                                        CUDA:0 (Tesla T4, 14912MiB)
YOLO26m summary (fused): 130 layers, 20,350,994 parameters, 0 gradients, 68.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.2±0.3 ms, read: 15.0±13.1 MB/s, size: 145.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/labels... 429